# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PalSoham/flyrank-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I am choosing **Lane 2: Refresh / Content Opportunity Scoring**.

The question this lane answers is: *which pages in a content inventory should be reviewed first for refresh, expansion, protection, pruning, or monitoring?* A content team has limited editorial capacity every week. Without a principled ranking, reviewers either pick pages at random or rely on gut feel. Both waste time and miss real problems. The starter dataset already ships all the signals this lane needs — 90-day search impressions, clicks, sessions, CTR, position, engagement rate, scroll rate, content age, and freshness — so I can show a complete end-to-end workflow (transparent baseline → learned model → ranked action queue) without waiting for warehouse access. The starter pipeline also provides a working random-forest result with client-holdout validation that I can inspect, challenge, and improve over the next seven weeks. That combination — a clear decision, real data already in hand, a baseline to beat, and a final artifact a reviewer could actually use — makes Lane 2 the strongest starting point for a meaningful capstone.

In [1]:
import pandas as pd
import os

# Locate the starter dataset — works from Colab, Jupyter, or the repo root
for candidate in [
    '../../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv',
]:
    if os.path.exists(candidate):
        DATA_PATH = candidate
        break

df = pd.read_csv(DATA_PATH)
print('Starter dataset shape:', df.shape)
print('Unique clients:', df['client_id'].nunique())
print('Columns (count):', len(df.columns))


Starter dataset shape: (30000, 44)
Unique clients: 32
Columns (count): 44


## 2. The question: decision, action, cost of a wrong call

**Decision this work improves:**  
Which content pages should an editorial team review *first* in a given sprint, given limited reviewer time?

**Unit of analysis:** one content page (one row in the starter dataset = one pseudonymized content item with 90 days of trailing search and engagement metrics).

**Output:** a ranked action queue — each page receives a score (0-100), a suggested action (`refresh`, `refresh_and_review_ctr`, `refresh_and_review_engagement`, `expand_and_refresh`, or `monitor`), and one or more human-readable reason codes explaining the score.

**Who acts on it and what do they do:**  
A content strategist or SEO editor opens the ranked queue at the start of the week, reads the top-20 entries, confirms that the reason codes make editorial sense, and schedules those pages for content review. They do not act automatically — the queue is a reviewer aid, not an auto-publish system.

**Cost of a wrong call:**  
- *False positive (rank a page that does not need work):* wasted editor hours reviewing a healthy page. Low cost per error, but adds up quickly if the list is noisy.  
- *False negative (miss a page that genuinely needs attention):* a declining, high-traffic page continues losing visibility undetected. Potentially high cost — missed traffic, clients lose confidence in the platform.  
Because missing real problems is more expensive than reviewing an extra healthy page, I will optimise for **recall among high-impression pages** while accepting moderate false-positive rates at the tail.

**Why data / ML helps at all:**  
A fixed rule (e.g. 'flag any page older than 180 days with impressions > 500') misses the interaction between age, freshness, position trend, CTR gap, and engagement. Those signals are real but entangled in different ways for different content types. The starter pipeline already shows that a random forest (Precision@50 = 0.74) beats the baseline rules (Precision@50 = 0.24) by 3x on the same slice — evidence that a learned ranking is worth the extra complexity over a hand-written if-statement.

In [2]:
# Supporting evidence for the framing
# Note: trend_direction and trend_pct are NEVER used as features (label source)
decline_mask = df['trend_direction'] == 'down'
n_declining = decline_mask.sum()
total = len(df)
print(f'Declining pages: {n_declining} of {total} ({n_declining/total*100:.1f}%)')

# Baseline rule comparison (from verified outputs/model_report.md)
results = pd.DataFrame({
    'Method': ['baseline_rules', 'logistic_regression', 'decision_tree', 'random_forest'],
    'ROC AUC': [0.627, 0.700, 0.742, 0.750],
    'Avg Precision': [0.468, 0.522, 0.575, 0.618],
    'Precision@50': [0.240, 0.400, 0.540, 0.740],
})
print(results.to_string(index=False))
lift = 0.740 / 0.240
print(f'Random forest Precision@50 is {lift:.1f}x better than baseline rules')


Declining pages: 16262 of 30000 (54.2%)
               Method  ROC AUC  Avg Precision  Precision@50
       baseline_rules    0.627          0.468          0.24
  logistic_regression    0.700          0.522          0.40
        decision_tree    0.742          0.575          0.54
        random_forest    0.750          0.618          0.74
Random forest Precision@50 is 3.1x better than baseline rules


## 3. Quick look at the data (2-3 real numbers)

The code cell below shows three real numbers that make this lane worth seven weeks.

1. **54.2% of pages are labelled declining** (16,262 of 30,000). This is not a niche edge case — it is the majority of the inventory. A ranked queue that surfaces the worst-of-the-worst is genuinely useful.

2. **5,014 pages combine high visibility (>=500 impressions/90d) with a declining trend.** Their median impression count is 1,627 per 90 days. These are not tiny ghost pages — they have real search exposure and are actively losing it. Reviewing all 5,014 at once is impossible; a ranked priority score is exactly what the team needs.

3. **2,920 declining pages already hold a page-1 Google position (avg_position <= 10).** A page-1 ranking is hard to earn and easy to lose. Catching decay *before* a page drops off the first page is a high-value, time-sensitive decision — and one a reviewer alone cannot systematically track across thousands of pages without a queue.

In [3]:
# Number 1: decline rate
decline_mask = df['trend_direction'] == 'down'
n_declining = decline_mask.sum()
total = len(df)
print(f'=== Number 1 ===')
print(f'Declining pages: {n_declining} of {total} ({n_declining/total*100:.1f}%)')

# Number 2: high-volume declining pages
high_vol_declining = df[decline_mask & (df['impressions_90d'] >= 500)]
print(f'\n=== Number 2 ===')
print(f'Pages with >=500 impressions/90d AND declining: {len(high_vol_declining)}')
median_imp = int(high_vol_declining['impressions_90d'].median())
print(f'Median impressions of this group: {median_imp}')

# Number 3: position-tier breakdown for declining pages
print(f'\n=== Number 3 ===')
tier_counts = df[decline_mask]['position_tier'].value_counts()
print(tier_counts)
page1_n = int(tier_counts.get('page_1', 0))
print(f'\nDeclining pages on page 1 (position_tier == page_1): {page1_n}')


=== Number 1 ===
Declining pages: 16262 of 30000 (54.2%)

=== Number 2 ===
Pages with >=500 impressions/90d AND declining: 5014
Median impressions of this group: 1627

=== Number 3 ===
position_tier
deep          5777
no_data       3872
page_1        2920
striking      2027
page_3_5       878
top_3          788
Name: count, dtype: int64

Declining pages on page 1 (position_tier == page_1): 2920


## 4. Careful words: what I can and can't claim

### What I *can* claim (careful, observed, directional language)

- **Observed association:** Pages with these signal combinations (declining trend, high impressions, low CTR relative to position) were disproportionately represented among the top-ranked review candidates in the training slice.
- **Decision-support:** This ranked queue is a reviewer aid. A page scoring in the top decile is a candidate for editorial review — not a guarantee that refreshing it will recover traffic.
- **Directional improvement:** Across the starter slice, a learned ranking placed roughly 3x more true declining pages in its top 50 than the baseline rule set did. If this pattern holds on the full warehouse data after proper validation, the queue would direct editor time more efficiently than the current rule.
- **Measurable benchmark:** I will report Precision@K (K = 20, 50), ROC-AUC, and average precision, using client-holdout or time-aware validation so that test clients were not seen during training.

### What I *cannot* claim

- **Causality:** I cannot claim that refreshing a page will cause traffic to recover. The data is observational — it shows what happened, not why. Proving a causal effect would require a controlled experiment that this dataset does not support.
- **Google algorithm factors:** The signals I use are measurements of what Google returned and what users did — not evidence of how Google's algorithm weights those signals internally.
- **Prediction at individual page level:** The model ranks pages by probability of fitting the declining pattern. A high score means the page *looks like* a declining page in this data; it does not mean the specific page will definitely decline further.
- **Generalisation without validation:** Results from the 30,000-row starter slice may not hold across the full 519,606-item content inventory in the warehouse. Full validation is required before broader claims.
- **AI citations or AI rankings:** The `ai_sessions_90d` column measures click-throughs from AI assistant tools to the page — not citations, not rankings, not visibility inside any AI product.

In [4]:
# Sanity check: confirm key data properties from the data dictionary
print('=== Sanity check ===')
print(f'Total rows: {len(df)}')
print(f'Unique clients: {df["client_id"].nunique()}')
print(f'Rows with avg_position == 0 (no position data, per data dictionary): {(df["avg_position"] == 0).sum()}')
print(f'Rows with missing trend_pct (prev_30d impressions = 0): {df["trend_pct"].isna().sum()}')
ctr_max = df['ctr'].max()
print(f'CTR max value: {ctr_max} (x100 percentage — 100 = 100%, per data dictionary)')
print('No raw URLs, client names, or query text anywhere in this notebook.')


=== Sanity check ===
Total rows: 30000
Unique clients: 32
Rows with avg_position == 0 (no position data, per data dictionary): 1205
Rows with missing trend_pct (prev_30d impressions = 0): 3388
CTR max value: 100.0 (x100 percentage — 100 = 100%, per data dictionary)
No raw URLs, client names, or query text anywhere in this notebook.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.